# TESSCH Recommendation Engine — Exploratory Data Analysis & Similarity Validation

**Goal:** Build and evaluate a behavior-based product recommendation system for TESSCH's modular sneaker storefront.

### Core Concepts Explored:
1. **Implicit Feedback & Dwell Time:** Measuring viewing duration as an implicit signal of interest without requiring explicit ratings.
2. **Personalized 'Picked For You':** Aggregated dwell time rankings per individual user with recency weighting.
3. **Cross-User 'You Might Also Like':** Assembling a user-item dwell matrix, applying row normalization, and computing item-to-item cosine similarity.
4. **Coherence & Stability Evaluation:** Validating whether recommendations reflect true semantic clusters compared against a random baseline.

In [ ]:
import sqlite3
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Display settings
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

DB_PATH = Path('../data/tessch.db')
print(f"Database exists: {DB_PATH.exists()}")

## 1. Data Ingestion & Catalogue Exploration

Let's query the SQLite database to load products, user profiles, and raw behavioral dwell events.

In [ ]:
conn = sqlite3.connect(DB_PATH)
products_df = pd.read_sql_query("SELECT * FROM products", conn)
events_df = pd.read_sql_query("SELECT * FROM user_events", conn)
users_df = pd.read_sql_query("SELECT * FROM users", conn)
conn.close()

print(f"Products: {len(products_df):,}")
print(f"Users: {len(users_df):,}")
print(f"Total Events: {len(events_df):,}")
print(f"Dwell Views: {len(events_df[events_df['event_type'] == 'product_view']):,}")

products_df[['product_id', 'product_name', 'category', 'style', 'color', 'price']].head(8)

## 2. Dwell Time Distribution & Data Cleansing

Raw browser dwell time exhibits heavy right-skew (exponential/log-normal distribution). Two crucial guards protect our system:
- **Minimum Dwell Floor (1.0s):** Discard immediate bounces and mis-clicks.
- **Maximum Dwell Ceiling (1800.0s / 30m):** Clamp abandoned tabs so a single 3-hour session does not dwarf an entire browsing history.

In [ ]:
views = events_df[events_df['event_type'] == 'product_view']['duration_seconds'].dropna()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].hist(views, bins=40, color='#2f5f96', edgecolor='white')
ax[0].set_title('Raw Dwell Time Distribution (Seconds)')
ax[0].set_xlabel('Duration (s)')
ax[0].set_ylabel('Observation Count')

ax[1].hist(np.log1p(views), bins=40, color='#1f7a74', edgecolor='white')
ax[1].set_title('Log-Transformed Dwell Time ln(1 + s)')
ax[1].set_xlabel('Log Duration')
ax[1].set_ylabel('Observation Count')

plt.tight_layout()
plt.show()

print(f"Dwell Summary Stats:\n{views.describe()}")

## 3. User Activity Inequality & The Need for Normalization

In any e-commerce platform, user browsing volume follows a power law: a small subset of power-users accounts for a disproportionate share of total attention.

Let's measure the user activity distribution using the **Gini Coefficient** and plot the **Lorenz Curve**.

In [ ]:
user_dwell = events_df[events_df['event_type'] == 'product_view'].groupby('user_id')['duration_seconds'].sum().sort_values()

# Calculate Lorenz Curve
n = len(user_dwell)
cum_users = np.linspace(0, 1, n)
cum_dwell = np.cumsum(user_dwell) / user_dwell.sum()

# Gini Coefficient
gini = 1 - 2 * np.trapezoid(cum_dwell, cum_users)

plt.figure(figsize=(7, 6))
plt.plot(cum_users, cum_dwell, label=f'Empirical Lorenz Curve (Gini = {gini:.2f})', color='#b8322b', lw=2.5)
plt.plot([0, 1], [0, 1], '--', color='grey', label='Perfect Equality (Gini = 0)')
plt.fill_between(cum_users, cum_users, cum_dwell, alpha=0.15, color='#b8322b')
plt.title('User Dwell Time Concentration (Lorenz Curve)')
plt.xlabel('Cumulative Fraction of Users')
plt.ylabel('Cumulative Fraction of Dwell Time')
plt.legend(loc='upper left')
plt.grid(True)
plt.show()

## 4. Constructing the User-Item Matrix & L2 Normalisation

We assemble the matrix $R \in \mathbb{R}^{M \times N}$ where $R_{u, i}$ is the total dwell time user $u$ spent on product $i$.

### L2 Row Normalisation:
$$\hat{R}_{u, i} = \frac{R_{u, i}}{\sqrt{\sum_{j} R_{u, j}^2}}$$

This ensures that each user's row vector has unit Euclidean length ($\|\hat{\mathbf{r}}_u\|_2 = 1$). A user who spent 10 hours browsing does not overshadow a user who spent 10 minutes.

In [ ]:
valid_views = events_df[(events_df['event_type'] == 'product_view') & (events_df['duration_seconds'] >= 1.0)]

# Filter users with at least 2 distinct products viewed
user_prod_counts = valid_views.groupby('user_id')['product_id'].nunique()
qualifying_users = user_prod_counts[user_prod_counts >= 2].index
valid_views = valid_views[valid_views['user_id'].isin(qualifying_users)]

# Pivot into User-Item Matrix
pivot = valid_views.groupby(['user_id', 'product_id'])['duration_seconds'].sum().unstack(fill_value=0.0)

# Reindex with all product IDs
all_pids = sorted(products_df['product_id'].unique())
pivot = pivot.reindex(columns=all_pids, fill_value=0.0)

raw_matrix = pivot.values
# Apply L2 row normalisation
row_norms = np.linalg.norm(raw_matrix, axis=1, keepdims=True)
row_norms[row_norms == 0.0] = 1.0
normalized_matrix = raw_matrix / row_norms

print(f"User-Item Matrix Shape: {normalized_matrix.shape[0]} users x {normalized_matrix.shape[1]} products")
density = np.count_nonzero(normalized_matrix) / (normalized_matrix.shape[0] * normalized_matrix.shape[1])
print(f"Matrix Density: {density:.2%}")

## 5. Item-to-Item Cosine Similarity Computation

We compute pairwise cosine similarity across the **columns** of the normalized matrix:

$$\text{Sim}(\mathbf{p}_i, \mathbf{p}_j) = \frac{\mathbf{p}_i^T \mathbf{p}_j}{\|\mathbf{p}_i\|_2 \|\mathbf{p}_j\|_2}$$

This measures how closely the viewing profiles for two sneakers align across all shoppers.

In [ ]:
item_vectors = normalized_matrix.T
sim_matrix = cosine_similarity(item_vectors)
np.fill_diagonal(sim_matrix, 1.0)

# Zero out cold items
dead_items = ~item_vectors.any(axis=1)
sim_matrix[dead_items, :] = 0.0
sim_matrix[:, dead_items] = 0.0

# Heatmap for top 25 active products
subset_size = 25
plt.figure(figsize=(9, 8))
plt.imshow(sim_matrix[:subset_size, :subset_size], cmap='viridis', vmin=0, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title(f'Item-to-Item Cosine Similarity Matrix (Top {subset_size} Products)')
plt.xticks(range(subset_size), all_pids[:subset_size], rotation=90, fontsize=8)
plt.yticks(range(subset_size), all_pids[:subset_size], fontsize=8)
plt.grid(False)
plt.show()

## 6. Coherence Validation: Category Lift vs Chance

Because the recommender only observes dwell time and has zero access to product metadata (category, style, color), checking whether similar items share attributes serves as an objective validation.

In [ ]:
product_cat_map = dict(zip(products_df['product_id'], products_df['category']))

top1_matches = []
topk_matches = []
k = 5

for i, pid in enumerate(all_pids):
    if dead_items[i]:
        continue
    scores = sim_matrix[i].copy()
    scores[i] = -1.0 # exclude self
    top_indices = np.argsort(-scores)[:k]
    
    src_cat = product_cat_map.get(pid)
    top_cats = [product_cat_map.get(all_pids[idx]) for idx in top_indices if scores[idx] > 0]
    
    if top_cats:
        top1_matches.append(top_cats[0] == src_cat)
        topk_matches.extend([c == src_cat for c in top_cats])

# Random Baseline
cat_counts = products_df['category'].value_counts(normalize=True)
random_baseline = (cat_counts ** 2).sum()

top1_rate = np.mean(top1_matches)
topk_rate = np.mean(topk_matches)
lift = topk_rate / random_baseline

print("=" * 50)
print(f"Top-1 Category Match Rate:  {top1_rate:.1%}")
print(f"Top-{k} Category Match Rate:  {topk_rate:.1%}")
print(f"Random Baseline (Chance):   {random_baseline:.1%}")
print(f"Coherence Lift over Chance: {lift:.2f}x")
print("=" * 50)

## 7. Worked Example Recommendations

Let's inspect sample recommendations for a flagship modular sneaker to confirm sensible suggestions.

In [ ]:
def show_recommendations(target_pid, top_n=4):
    if target_pid not in all_pids:
        print(f"Product {target_pid} not found.")
        return
    idx = all_pids.index(target_pid)
    scores = sim_matrix[idx].copy()
    scores[idx] = -1.0
    top_idx = np.argsort(-scores)[:top_n]
    
    target = products_df[products_df['product_id'] == target_pid].iloc[0]
    print(f"\nTarget Product: [{target['product_id']}] {target['product_name']} ({target['category']} / {target['style']} / ₹{target['price']:,.0f})")
    print("-" * 70)
    
    for rank, i in enumerate(top_idx, 1):
        sim_score = scores[i]
        rec = products_df[products_df['product_id'] == all_pids[i]].iloc[0]
        same_cat = "(Same Category)" if rec['category'] == target['category'] else "(Cross-Category)"
        print(f"{rank}. Score: {sim_score:.3f} | [{rec['product_id']}] {rec['product_name']:<24} | {rec['category']:<10} | ₹{rec['price']:<7,.0f} {same_cat}")

show_recommendations('T001', top_n=5)
show_recommendations('T015', top_n=5)

## 8. Summary & Key Takeaways

- **Dwell Time is a Rich Implicit Signal:** Continuous attention captures strong preference clusters without intrusive rating modals.
- **L2 Normalisation is Essential:** Row scaling prevents hyper-active browsers from skewing cross-user similarities while maintaining high category lift.
- **Cold-Start Strategy:** Products and users with insufficient viewing history gracefully fall back to popularity rankings with transparent provenance.